# Carga de clima (Open-Meteo)

Usa o módulo `clima` (`src/clima`). Antes: `pip install -r requirements.txt` na raiz do projeto e executar o `01_ibge_localidades.ipynb`.

A carga também roda fora do notebook: `python -m clima --help`.

In [ ]:
from pathlib import Path

from clima.cidades import carregar_municipios, representativas
from clima.db import consultar
from clima.open_meteo import atualizar_clima

# O notebook roda dentro de notebooks/; o banco fica em dados/, um nível acima.
DB = Path("..") / "dados" / "warehouse.duckdb"

In [ ]:
municipios = carregar_municipios(DB, "SP")
print(len(municipios), municipios[["lat", "lng", "fuso_horario"]].isna().sum().to_dict()) 

# Uma cidade por região imediata (economiza o limite de uso da API).
# Para todos os municípios, use `cidades = municipios` (leva muitos dias na cota gratuita).
cidades = representativas(municipios)
print(len(cidades), "cidades na carga")

In [ ]:
# Retomável: se parar por limite de uso, rode de novo mais tarde.
# Para um teste rápido: atualizar_clima(cidades.head(5), DB)
atualizar_clima(cidades, DB)

In [ ]:
consultar("""
    select fonte, count(*) as linhas, count(distinct ibge_id) as cidades,
           min(data) as de, max(data) as ate
    from bronze.clima_diario group by 1
""", DB)

In [ ]:
consultar("select * from bronze.clima_diario order by ibge_id, data limit 10", DB)